In [1]:
import re
from typing import Any
import joblib

import pandas as pd
from nltk import download
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

In [2]:
download("punkt", quiet=True)
download("punkt_tab", quiet=True)
download("stopwords", quiet=True)
download("wordnet", quiet=True)

True

In [3]:
df = pd.read_csv(r'../data/not_seen_emails.csv')

In [4]:
df.sample(3)

,email,label
1,['Subject: the new power company ; reserved sh...,0
3,"['Subject: less time , less effort but better ...",1
0,['Subject: meeting reminder \nthe meeting to ...,0


# 1. Pre procesamiento

In [5]:
def clean_email(text):
    """Limpieza específica para correos Enron."""
    text = str(text)
    # Remover headers típicos de correo
    text = re.sub(r"Subject:.*?\n", " ", text)
    text = re.sub(r"From:.*?\n", " ", text)
    text = re.sub(r"To:.*?\n", " ", text)
    # Remover separadores de forwarded/reply
    text = re.sub(r"-{3,}.*?-{3,}", " ", text)
    # Remover URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    # Remover emails
    text = re.sub(r"\S+@\S+", " ", text)
    # Remover caracteres especiales y números
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    # Normalizar espacios
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

In [6]:
stop_words = set(stopwords.words("english"))

def preprocess(text: str, stop_words: set = stop_words, lemmatizer: Any = WordNetLemmatizer()):
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

In [7]:
df['email_clean'] = df.apply(lambda row: clean_email(row['email']),axis=1)

In [8]:
df['email_tokens'] = df.apply(lambda row: preprocess(row['email_clean']), axis=1)

# 2. Inferencia

In [9]:
pipeline = joblib.load(r'../models/nb_pipeline.joblib')

In [10]:
y_pred = pipeline.predict(df['email_tokens'])

# 3. Resultados

In [11]:
df['label_pred'] = y_pred

In [12]:
df[['email', 'label', 'label_pred']]

,email,label,label_pred
0,['Subject: meeting reminder \nthe meeting to ...,0,0
1,['Subject: the new power company ; reserved sh...,0,0
2,['Subject: delivery failure : user antonio _ o...,1,1
3,"['Subject: less time , less effort but better ...",1,1
4,['Subject: cartier piaget - expensive look not...,1,1
5,"[""Subject: re : enron credit model docs for th...",0,0
6,['Subject: why pay when it \' s free ? abw 9 x...,1,1
